<a href="https://colab.research.google.com/github/EmanHrustemovic/FlyRank-AI-Intership-ML-Track-/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
#Setting up

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('eman')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

query = """
WITH daily AS (
    SELECT * FROM read_parquet('""" + rel + """/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
),
early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impr_early,
        SUM(gsc_clicks) AS clicks_early,
        SUM(gsc_sum_position) AS sum_pos_early
    FROM daily WHERE report_date <= DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
late AS (
    SELECT content_hash_id,
        SUM(gsc_impressions) AS impr_late,
        SUM(gsc_sum_position) AS sum_pos_late
    FROM daily WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id
),
labeled AS (
    SELECT
        e.client_hash_id, e.content_hash_id, e.impr_early, e.clicks_early,
        (e.sum_pos_early * 1.0 / NULLIF(e.impr_early,0)) AS avg_position_early,
        (e.clicks_early * 1.0 / NULLIF(e.impr_early,0)) AS ctr_early,
        CASE WHEN (l.sum_pos_late * 1.0 / NULLIF(l.impr_late,0))
             > (e.sum_pos_early * 1.0 / NULLIF(e.impr_early,0)) THEN 1 ELSE 0 END AS is_declining_label
    FROM early e JOIN late l ON e.content_hash_id = l.content_hash_id
    WHERE e.impr_early > 0 AND l.impr_late > 0
)
SELECT lb.*, dc.word_count, DATE '2026-03-15' - dc.content_updated_date AS days_since_update
FROM labeled lb
LEFT JOIN read_parquet('""" + rel + """/dim_content.parquet') dc ON lb.content_hash_id = dc.content_hash_id
"""

df2 = con.sql(query).df().dropna()
X2 = df2[['impr_early', 'avg_position_early', 'ctr_early', 'word_count', 'days_since_update']]
y2 = df2['is_declining_label']
groups = df2['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X2, y2, groups))
X_train_g, y_train_g = X2.iloc[train_idx], y2.iloc[train_idx]

logreg_g = LogisticRegression(max_iter=1000).fit(X_train_g, y_train_g)
print("Setup complete:", df2.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete: (92148, 9)


In [2]:
# Retrain final model on full grouped-split training data for the playbook
logreg_final = logreg_g  # reuse the client-grouped model from w06 (most honest available)

# Score all pages
all_scores = logreg_final.predict_proba(X2)[:, 1]
coefs = dict(zip(X2.columns, logreg_final.coef_[0]))

# Per-row contribution = coefficient * feature value, find the dominant driver
contributions = X2.copy()
for col in X2.columns:
    contributions[col] = X2[col] * coefs[col]

dominant_feature = contributions.idxmax(axis=1)

reason_map = {
    'avg_position_early': 'WEAK_POSITION',
    'ctr_early': 'CTR_UNDERPERFORM',
    'days_since_update': 'STALE_CONTENT',
    'impr_early': 'LOW_VOLUME',
    'word_count': 'THIN_CONTENT',
}

playbook = df2.copy()
playbook['score'] = all_scores
playbook['reason_code'] = dominant_feature.map(reason_map)
playbook['action'] = playbook['reason_code'].map({
    'WEAK_POSITION': 'improve-ranking',
    'CTR_UNDERPERFORM': 'fix-ctr',
    'STALE_CONTENT': 'refresh',
    'LOW_VOLUME': 'monitor',
    'THIN_CONTENT': 'expand-content',
})

playbook = playbook.sort_values('score', ascending=False)
playbook[['content_hash_id', 'score', 'reason_code', 'action']].head(20)

,content_hash_id,score,reason_code,action
31058,content_5a6ff506ec9ed4a1,0.663163,STALE_CONTENT,refresh
58285,content_978b74182e9983a1,0.663159,STALE_CONTENT,refresh
52132,content_feac82bbcd4578f0,0.663009,STALE_CONTENT,refresh
58570,content_9ae595d8da70b3d8,0.662974,STALE_CONTENT,refresh
34286,content_7e9880d1405fe0f5,0.662772,STALE_CONTENT,refresh
31603,content_60a6fd1ad5abe943,0.662772,STALE_CONTENT,refresh
133471,content_320cb2675eb003e3,0.662676,STALE_CONTENT,refresh
57886,content_9352fa643a28d81a,0.662654,STALE_CONTENT,refresh
63907,content_d5d55b528496fab8,0.662600,STALE_CONTENT,refresh
65459,content_e763574cb54d7298,0.662457,STALE_CONTENT,refresh


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X2_scaled = pd.DataFrame(scaler.fit_transform(X2), columns=X2.columns, index=X2.index)

contributions_scaled = X2_scaled.copy()
for col in X2.columns:
    contributions_scaled[col] = X2_scaled[col] * coefs[col]

dominant_feature_scaled = contributions_scaled.idxmax(axis=1)

playbook['reason_code'] = dominant_feature_scaled.map(reason_map)
playbook['action'] = playbook['reason_code'].map({
    'WEAK_POSITION': 'improve-ranking',
    'CTR_UNDERPERFORM': 'fix-ctr',
    'STALE_CONTENT': 'refresh',
    'LOW_VOLUME': 'monitor',
    'THIN_CONTENT': 'expand-content',
})

playbook = playbook.sort_values('score', ascending=False)
print(playbook['reason_code'].value_counts())
playbook[['content_hash_id', 'score', 'reason_code', 'action']].head(20)

reason_code
WEAK_POSITION       63197
STALE_CONTENT       13508
THIN_CONTENT         8276
LOW_VOLUME           4330
CTR_UNDERPERFORM     2837
Name: count, dtype: int64


,content_hash_id,score,reason_code,action
31058,content_5a6ff506ec9ed4a1,0.663163,WEAK_POSITION,improve-ranking
58285,content_978b74182e9983a1,0.663159,WEAK_POSITION,improve-ranking
52132,content_feac82bbcd4578f0,0.663009,WEAK_POSITION,improve-ranking
58570,content_9ae595d8da70b3d8,0.662974,WEAK_POSITION,improve-ranking
34286,content_7e9880d1405fe0f5,0.662772,WEAK_POSITION,improve-ranking
31603,content_60a6fd1ad5abe943,0.662772,WEAK_POSITION,improve-ranking
133471,content_320cb2675eb003e3,0.662676,WEAK_POSITION,improve-ranking
57886,content_9352fa643a28d81a,0.662654,WEAK_POSITION,improve-ranking
63907,content_d5d55b528496fab8,0.662600,WEAK_POSITION,improve-ranking
65459,content_e763574cb54d7298,0.662457,WEAK_POSITION,improve-ranking


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.